# Custom GB grid LCA

Builds a custom GB electricity-mix foreground activity per half-hour
timeslice, then runs the foreground H2-technology LCAs with that mix
substituted for direct electricity inputs.

All settings live in [dashboard_config.py](dashboard_config.py).
Toggle with `RUN_GRID_SCENARIO_LCA`; choose `METHOD_MODE = "exact"` or `"cheap"`.

The output CSV is consumed by the dashboard notebook for plotting.

In [1]:
# Run flags / global settings come from dashboard_config.py.
# Ecoinvent queries + candidate indexes for THIS notebook live below.
from dashboard_config import *
import dashboard_config as cfg
import lca_helpers as H

print_dashboard()
print()
ei, bio, fg_db, method = H.setup_brightway()

Master Dashboard
----------------
Project:                 hydrogen-smr
Foreground DB:           hydrogen foreground
Build foreground DB:     True
Run reference LCA:       False
Run grid scenario LCA:   True
Run wind/grid LCA:       True
Grid method:             cheap | loss factor: 1.0316426921769999
Selected grid techs:     ['AE operation']
Wind/grid mode:          both (blended + switching) | electrolyser(s): ['AE operation']

Current Brightway project: hydrogen-smr
Using ecoinvent database: ecoinvent-3.9.1-apos
Using biosphere database: ecoinvent-3.9.1-biosphere
Using foreground database: hydrogen foreground
Using LCIA method: ('ecoinvent-3.9.1', 'IPCC 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')


In [2]:
selected_grid_source = getattr(cfg, "GRID_SOURCE_NORMALIZED", cfg.normalise_grid_data_source(getattr(cfg, "GRID_DATA_SOURCE", "csv")))
if selected_grid_source != "csv":
    raise SystemExit(
        f"Skipping 3.custom_grid.ipynb because GRID_DATA_SOURCE={getattr(cfg, 'GRID_DATA_SOURCE', None)!r}. "
        "Set GRID_DATA_SOURCE='csv' or '4' to use this notebook."
    )

if not RUN_GRID_SCENARIO_LCA:
    raise SystemExit(
        "RUN_GRID_SCENARIO_LCA is False in dashboard_config.py. "
        "Set it True and re-run this notebook."
    )

if fg_db is None:
    raise SystemExit(
        "Foreground database not found. Run tech_lca_foreground.ipynb first "
        "(set RUN_BUILD_FOREGROUND_DATABASE=True in dashboard_config.py)."
    )


## Custom GB electricity mix — ecoinvent template

Three dicts that fully describe how the custom GB grid is built:

| dict | what it picks |
|---|---|
| `GRID_CANDIDATE_INDEX` | one ecoinvent process per generation/import component |
| `GRID_INFRASTRUCTURE`  | constant non-energy technosphere inputs per kWh (transmission, SF6) |
| `GRID_EMISSIONS`       | constant non-energy biosphere emissions per kWh (SF6, N2O, ozone) |

Each component lists a `query` (passed to `ei.search(...)` /  `bio.search(...)`)
and a `candidate_index` (which hit to use). Set `SHOW_GRID_ECOINVENT_CANDIDATES`
in [dashboard_config.py](dashboard_config.py) to print every candidate list, then
edit the integers below to switch processes.

In [3]:
# --- Generation / import components ---------------------------------------
# TECH_SPECS (in lca_helpers.py) supplies the queries; choose the candidate.
GRID_CANDIDATE_INDEX = {
    # Conventional generation
    "GAS":                3,
    "COAL":               0,
    "NUCLEAR":            1,
    # Wind sub-technologies (composite, split by WIND_GROUP_WEIGHTS)
    "WIND_GT3_ONSHORE":   0,
    "WIND_13_OFFSHORE":   0,
    "WIND_13_ONSHORE":    0,
    "WIND_EMB":           0,
    # Hydro
    "HYDRO":              0,
    # Imports (composite, split by IMPORT_GROUP_WEIGHTS)
    "IMPORTS_FR":         0,
    "IMPORTS_IE":         0,
    "IMPORTS_NL":         4,
    # Other generation
    "BIOMASS":            0,
    "OTHER":              0,
    "SOLAR":              0,
    "STORAGE":            0,
}

# --- Constant non-energy technosphere inputs (per kWh) --------------------
GRID_INFRASTRUCTURE = {
    "TRANSMISSION_HV": {
        "query":            "market for transmission network electricity high voltage",
        "name_contains":    ["market for transmission network", "high voltage"],
        "name_excludes":    ["direct current", "maintenance"],
        "candidate_index":  0,
        "amount":           6.58e-9,
        "unit":             "kilometer",
    },
    "SF6_INPUT": {
        "query":            "market for sulfur hexafluoride, liquid",
        "name_contains":    "market for sulfur hexafluoride",
        "candidate_index":  0,
        "amount":           2.99e-9,
        "unit":             "kilogram",
    },
}

# --- Constant non-energy biosphere emissions (per kWh) --------------------
GRID_EMISSIONS = {
    "SF6_AIR": {
        "query":            "Sulfur hexafluoride",
        "candidate_index":  0,
        "categories":       ("air", "non-urban air or from high stacks"),
        "amount":           2.99e-9,
        "unit":             "kilogram",
    },
    "N2O_AIR": {
        "query":            "Dinitrogen monoxide",
        "candidate_index":  0,
        "categories":       ("air", "non-urban air or from high stacks"),
        "amount":           4.90e-8,
        "unit":             "kilogram",
    },
    "OZONE_AIR": {
        "query":            "Ozone",
        "candidate_index":  0,
        "categories":       ("air", "non-urban air or from high stacks"),
        "amount":           4.15e-8,
        "unit":             "kilogram",
    },
}

print("Custom GB electricity mix — candidate indexes")
print("---------------------------------------------")
for tech_key, idx in GRID_CANDIDATE_INDEX.items():
    print(f"  {tech_key:<22} candidate #{idx}")

print()
print("Constant non-energy inputs (per kWh)")
print("------------------------------------")
for k, spec in GRID_INFRASTRUCTURE.items():
    print(f"  infra / {k:<24} {spec['amount']:>10.3e} {spec.get('unit', ''):<10}"
          f" [{spec['query']!r}, candidate #{spec.get('candidate_index', 0)}]")
for k, spec in GRID_EMISSIONS.items():
    if float(spec.get("amount", 0)) <= 0:
        continue
    cats = spec.get("categories") or "(any)"
    print(f"  emis  / {k:<24} {spec['amount']:>10.3e} {spec.get('unit', ''):<10}"
          f" [{spec['query']!r}, categories={cats}]")
# Show candidate lists when this notebook runs (set False to suppress).
SHOW_GRID_ECOINVENT_CANDIDATES = True


Custom GB electricity mix — candidate indexes
---------------------------------------------
  GAS                    candidate #3
  COAL                   candidate #0
  NUCLEAR                candidate #1
  WIND_GT3_ONSHORE       candidate #0
  WIND_13_OFFSHORE       candidate #0
  WIND_13_ONSHORE        candidate #0
  WIND_EMB               candidate #0
  HYDRO                  candidate #0
  IMPORTS_FR             candidate #0
  IMPORTS_IE             candidate #0
  IMPORTS_NL             candidate #4
  BIOMASS                candidate #0
  OTHER                  candidate #0
  SOLAR                  candidate #0
  STORAGE                candidate #0

Constant non-energy inputs (per kWh)
------------------------------------
  infra / TRANSMISSION_HV           6.580e-09 kilometer  ['market for transmission network electricity high voltage', candidate #0]
  infra / SF6_INPUT                 2.990e-09 kilogram   ['market for sulfur hexafluoride, liquid', candidate #0]
  emis  / SF6_AIR

## Load CSV and select timeslices

In [4]:
df = H.load_grid_csv()
run_rows, target_label = H.select_grid_rows(df)
row = run_rows.iloc[0]
TARGET_LABEL = str(row["DATETIME"]).replace(" ", "T")

print("Grid scenario selection")
print("-----------------------")
print("Mode:", GRID_TIME_MODE, "| rows:", len(run_rows), "| window:", target_label)
print("Reference timeslice for candidate checks:", row["DATETIME"])
print("Carbon intensity:", row.get("CARBON_INTENSITY", "n/a"), "g CO2/kWh")
print()
print(f"{'Technology':<15} {'MW':>10}   {'Share %':>8}")
print("-" * 38)
for col in H.FUEL_COLS:
    mw  = row.get(col, 0)
    pct = row.get(col + "_perc", 0)
    flag = "  <- negative/storage charging" if pct < 0 else ""
    try:
        print(f"{col:<15} {float(mw):>10.1f}   {float(pct):>7.2f}%{flag}")
    except Exception:
        pass

Loaded 305,447 rows from df_fuel_ckan.csv
Date range: 2009-01-01 00:00:00 → 2026-06-04 11:00:00
Grid scenario selection
-----------------------
Mode: year_average | rows: 576 | window: year2023_representative_576days
Reference timeslice for candidate checks: 2023-02-15 00:00:00
Carbon intensity: 133.0 g CO2/kWh

Technology              MW    Share %
--------------------------------------
GAS                 6153.0     22.70%
COAL                 232.0      0.90%
NUCLEAR             3534.0     13.00%
WIND                8468.0     31.20%
WIND_EMB            2036.0      7.50%
HYDRO                247.0      0.90%
IMPORTS             4340.0     16.00%
BIOMASS             1982.0      7.30%
OTHER                167.0      0.60%
SOLAR                  0.0      0.00%
STORAGE                0.0      0.00%


## Inspect ecoinvent candidates (optional)

Set `SHOW_GRID_ECOINVENT_CANDIDATES = True` at the top of this notebook (cell 4)
to print every candidate list (energy components + infrastructure + emissions).
Edit the `candidate_index` integers in the template cell above and re-run.


In [5]:
if SHOW_GRID_ECOINVENT_CANDIDATES:
    # Energy components (TECH_SPECS supplies the queries; GRID_CANDIDATE_INDEX picks the candidate)
    print(f"\n{'═'*70}\n  ENERGY COMPONENTS — ecoinvent candidates\n{'═'*70}")
    for tech, spec in H.TECH_SPECS.items():
        row_key = spec["row_key"]
        shares = run_rows[row_key + "_perc"] if (row_key + "_perc") in run_rows.columns else None
        max_share = float(shares.max()) if shares is not None else 0.0
        idx = int(GRID_CANDIDATE_INDEX.get(tech, 0))
        if max_share <= 0:
            print(f"\n{'━'*70}\n  {tech}  —  max share in window: {max_share:.2f}%  (SKIPPED)")
            continue
        print(f"\n{'━'*70}")
        print(f"  {tech}  —  max share: {max_share:.2f}%  (picked: candidate #{idx})")
        print(f"  Query: {spec['query']!r}\n")
        pool = list(ei.search(spec["query"]))[:100]
        for i, act in enumerate(pool):
            marker = "  ←" if i == idx else ""
            print(f"  {i:>2} | {act.get('name')} | ref: {act.get('reference product')}"
                  f" | unit: {act.get('unit')} | loc: {act.get('location')}{marker}")
        if idx >= len(pool):
            print(f"  ⚠ candidate_index {idx} out of range ({len(pool)})")

    # Constant non-energy infrastructure
    print(f"\n{'═'*70}\n  NON-ENERGY INFRASTRUCTURE — technosphere candidates\n{'═'*70}")
    for infra_key, spec in GRID_INFRASTRUCTURE.items():
        idx = int(spec.get("candidate_index", 0))
        amt = float(spec.get("amount", 0) or 0)
        print(f"\n{'━'*70}")
        print(f"  {infra_key}  —  amount per kWh: {amt:.3e} {spec.get('unit', '')}"
              f"  (picked: candidate #{idx})")
        print(f"  Query: {spec['query']!r}")
        raw_pool = list(ei.search(spec["query"]))[: int(spec.get("max_results", 100))]
        pool = H.filter_pool(raw_pool, spec)
        if not pool:
            print(f"  ⚠ No candidates after filters (raw hits: {len(raw_pool)})")
            continue
        for i, act in enumerate(pool):
            marker = "  ←" if i == idx else ""
            print(f"  {i:>2} | {act.get('name')} | ref: {act.get('reference product')}"
                  f" | unit: {act.get('unit')} | loc: {act.get('location')}{marker}")

    # Constant non-energy biosphere emissions
    print(f"\n{'═'*70}\n  NON-ENERGY EMISSIONS — biosphere candidates\n{'═'*70}")
    for emis_key, spec in GRID_EMISSIONS.items():
        idx = int(spec.get("candidate_index", 0))
        amt = float(spec.get("amount", 0) or 0)
        cats = spec.get("categories")
        print(f"\n{'━'*70}")
        print(f"  {emis_key}  —  amount per kWh: {amt:.3e} {spec.get('unit', '')}"
              f"  (picked: candidate #{idx})")
        print(f"  Query: {spec['query']!r}   categories filter: {cats}")
        raw_pool = list(bio.search(spec["query"]))[: int(spec.get("max_results", 100))]
        pool = H.filter_biosphere(raw_pool, spec)
        if not pool:
            print(f"  ⚠ No biosphere flows after filter (raw hits: {len(raw_pool)})")
            continue
        for i, flow in enumerate(pool):
            marker = "  ←" if i == idx else ""
            print(f"  {i:>2} | {flow.get('name')} | unit: {flow.get('unit')}"
                  f" | categories: {flow.get('categories')}{marker}")
else:
    print("SHOW_GRID_ECOINVENT_CANDIDATES is False — skipping candidate dump.")

# Resolve the ecoinvent activities for the reference timeslice
selected_processes = H.select_candidate_activities(row, GRID_CANDIDATE_INDEX)



══════════════════════════════════════════════════════════════════════
  ENERGY COMPONENTS — ecoinvent candidates
══════════════════════════════════════════════════════════════════════

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  GAS  —  max share: 60.90%  (picked: candidate #3)
  Query: 'electricity production, natural gas, combined cycle GB'

   0 | heat and power co-generation, natural gas, combined cycle power plant, 400MW electrical | ref: heat, district or industrial, natural gas | unit: megajoule | loc: GB
   1 | electricity production, natural gas, combined cycle power plant | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
   2 | heat and power co-generation, natural gas, combined cycle power plant, 400MW electrical | ref: electricity, high voltage | unit: kilowatt hour | loc: GB
   3 | electricity production, natural gas, conventional power plant | ref: electricity, high voltage | unit: kilowatt hour | loc: GB  ←
   4 | heat and po

## Build the custom GB electricity activity for the reference timeslice

In [6]:
new_act = H.build_custom_electricity_activity(
    row, selected_processes, fg_db,
    infrastructure=GRID_INFRASTRUCTURE, emissions=GRID_EMISSIONS,
)
print("Custom electricity activity:", new_act)

Deleted previous version: 'electricity market, GB custom mix (2023-02-15T00:00:00)'

Building: 'electricity market, GB custom mix (2023-02-15T00:00:00)'
Technology                 Amount   Process
──────────────────────────────────────────────────────────────────────────────────────
  GAS                      0.234183   electricity production, natural gas, conventional power plant [GB]
  COAL                     0.009285   electricity production, hard coal [GB]
  NUCLEAR                  0.134114   electricity production, nuclear, boiling water reactor [GB]
  WIND_EMB                 0.077373   electricity production, wind, <1MW turbine, onshore [GB]
  HYDRO                    0.009285   electricity production, hydro, run-of-river [GB]
  BIOMASS                  0.075310   heat and power co-generation, wood chips, 6667 kW, state-of-the-art 2014 [GB]
  OTHER                    0.006190   electricity production, oil [GB]
  WIND / WIND_GT3_ONSHORE   0.013743   electricity production, wind

## Resolve the selected H2 technologies

In [7]:
print("Selected technologies for LCA:", SELECTED_LCA_TECHS)
tech_activities = {}
for label in SELECTED_LCA_TECHS:
    code = H.H2_CODES.get(label)
    if code is None:
        print(f"  ⚠ Unknown technology label: {label}")
        continue
    try:
        import bw2data as bd
        tech_activities[label] = bd.get_activity((FOREGROUND_DB, code))
        print(f"  Found: {label}")
    except Exception:
        print(f"  ⚠  Not found: {label} (code={code!r})")

if not tech_activities:
    raise ValueError("No selected foreground activities were found. Check TECH_SELECTED.")

Selected technologies for LCA: ['AE operation']
  Found: AE operation


## Batch runner across all selected timeslices

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd

n_total = len(run_rows)
range_records = []

# Initialize cheap-method variables
source_lca_scores = {}
tech_decomposition = {}

if METHOD_MODE == "cheap":
    # Pre-calculate source LCA scores and technology decomposition for cheap method
    _gen = H.select_all_candidate_activities(GRID_CANDIDATE_INDEX)
    source_lca_scores = H.calculate_source_lca_scores(_gen, method)
    tech_decomposition = H.decompose_technology_scores(tech_activities, method, fg_db)

if METHOD_MODE == "exact":
    print(f"Running EXACT batch for {n_total} timeslice(s): {target_label}")
    print("Technologies:", list(tech_activities.keys()), "\n")
    for i, (_, csv_row) in enumerate(run_rows.iterrows()):
        ts = csv_row["DATETIME"]
        print(f"  [{i+1:>2}/{n_total}] {ts}", end="  ", flush=True)
        tmp_elec = None
        try:
            sel = H.select_candidate_activities(csv_row, GRID_CANDIDATE_INDEX)
            tmp_elec = H.build_custom_electricity_activity(
                csv_row, sel, fg_db,
                infrastructure=GRID_INFRASTRUCTURE, emissions=GRID_EMISSIONS,
                verbose=False,
            )
            record = {"datetime": ts, "carbon_intensity": csv_row.get("CARBON_INTENSITY", np.nan)}
            # Preserve REP_SEASON / REP_KIND tags if present (for year_average mode)
            if "REP_SEASON" in csv_row:
                record["REP_SEASON"] = csv_row["REP_SEASON"]
            if "REP_KIND" in csv_row:
                record["REP_KIND"] = csv_row["REP_KIND"]
            patch = {}
            for tech_label, tech_act in tech_activities.items():
                score, n_sub = H.run_lca_with_custom_elec(tech_act, tmp_elec, method)
                record[tech_label] = score
                patch[tech_label] = n_sub
            range_records.append(record)
            print("OK (" + ", ".join(f"{k}: {v} elec patched" for k, v in patch.items()) + ")")
        except Exception as e:
            print(f"SKIPPED ({type(e).__name__}: {e})")
        finally:
            if tmp_elec is not None:
                try: tmp_elec.delete()
                except Exception: pass
else:
    print(f"Running CHEAP batch for {n_total} timeslice(s): {target_label}")
    print("Technologies:", list(tech_decomposition.keys()), "\n")
    for i, (_, csv_row) in enumerate(run_rows.iterrows()):
        ts = csv_row["DATETIME"]
        elec_score, elec_amount_sum = H.custom_electricity_score_for_row(csv_row, source_lca_scores)
        record = {
            "datetime": ts,
            "carbon_intensity": csv_row.get("CARBON_INTENSITY", np.nan),
            "custom_electricity_score": elec_score,
            "custom_electricity_input_kwh_per_kwh": elec_amount_sum,
        }
        # Preserve REP_SEASON / REP_KIND tags if present (for year_average mode)
        if "REP_SEASON" in csv_row:
            record["REP_SEASON"] = csv_row["REP_SEASON"]
        if "REP_KIND" in csv_row:
            record["REP_KIND"] = csv_row["REP_KIND"]
        for tech_label, parts in tech_decomposition.items():
            record[tech_label] = parts["non_electricity_score"] + parts["electricity_kwh"] * elec_score
        range_records.append(record)
        if (i + 1) % 25 == 0 or i == 0 or i == n_total - 1:
            print(f"  [{i+1:>4}/{n_total}] {ts}  elec={elec_score:.6f} kg CO2eq/kWh")

if not range_records:
    raise RuntimeError("No timeslices were processed.")

range_df = pd.DataFrame(range_records).set_index("datetime")
print(f"\nCompleted {len(range_df)}/{n_total} timeslices ({METHOD_MODE} method).")
range_df.head()

Running CHEAP batch for 576 timeslice(s): year2023_representative_576days


NameError: name 'tech_decomposition' is not defined

## Optional validation: cheap vs exact

In [ ]:
import pandas as pd

if VALIDATE_CHEAP_METHOD:
    if not source_lca_scores:
        _gen = H.select_all_candidate_activities(GRID_CANDIDATE_INDEX)
        source_lca_scores = H.calculate_source_lca_scores(_gen, method)
    if not tech_decomposition:
        tech_decomposition = H.decompose_technology_scores(tech_activities, method, fg_db)

    validation_records = []
    for _, csv_row in run_rows.head(VALIDATION_N).iterrows():
        sel = H.select_candidate_activities(csv_row, GRID_CANDIDATE_INDEX)
        tmp_elec = H.build_custom_electricity_activity(
            csv_row, sel, fg_db,
            infrastructure=GRID_INFRASTRUCTURE, emissions=GRID_EMISSIONS,
            verbose=False,
        )
        try:
            elec_score, _ = H.custom_electricity_score_for_row(csv_row, source_lca_scores)
            for tech_label, tech_act in tech_activities.items():
                exact_score, n_sub = H.run_lca_with_custom_elec(tech_act, tmp_elec, method)
                parts = tech_decomposition[tech_label]
                cheap_score = parts["non_electricity_score"] + parts["electricity_kwh"] * elec_score
                validation_records.append({
                    "datetime": csv_row["DATETIME"], "technology": tech_label,
                    "exact_score": exact_score, "cheap_score": cheap_score,
                    "difference": cheap_score - exact_score,
                    "pct_difference": 100*(cheap_score-exact_score)/exact_score if exact_score else float("nan"),
                    "electricity_exchanges_replaced": n_sub,
                })
        finally:
            try: tmp_elec.delete()
            except Exception: pass

    validation_df = pd.DataFrame(validation_records)
    print(validation_df.to_string(index=False))
else:
    print("Validation skipped. Set VALIDATE_CHEAP_METHOD=True in dashboard_config.py.")

Validation skipped. Set VALIDATE_CHEAP_METHOD=True in dashboard_config.py.


## Export results CSV (consumed by the dashboard notebook)

In [ ]:
from pathlib import Path

# Mark the source so the adaptive dashboard cannot accidentally load stale 4.1 outputs.
range_df = range_df.copy()
range_df["grid_data_source"] = "csv"

# Make the selected rows available to downstream notebooks when run via
# 1.dashboard_lca_adaptive.ipynb in a shared kernel.
DASHBOARD_GRID_RUN_ROWS = run_rows.copy()
DASHBOARD_GRID_TARGET_LABEL = target_label
DASHBOARD_GRID_DATA_SOURCE = "csv"

output_dir = Path(GRID_OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)
safe_window = str(target_label).replace(" ", "T").replace(":", "-").replace("->", "_to_")
csv_out = output_dir / f"custom_grid_lca_csv_{METHOD_MODE}_{safe_window}.csv"
range_df.to_csv(csv_out)
print("Saved CSV custom-grid LCA results to:")
print(csv_out.resolve())


Saved custom-grid LCA results to:
C:\Users\ls1524\Documents\brightway\working copy\brightway-main\split up version\custom_grid_lca_outputs\custom_grid_lca_exact_year2023_representative_12days.csv


## Seasonal wind-contribution analysis

For a chosen `ANALYSIS_YEAR` (defaults to `GRID_YEAR` in
[dashboard_config.py](dashboard_config.py)), split the half-hourly CSV into
meteorological seasons (Winter = Dec/Jan/Feb, Spring = Mar–May,
Summer = Jun–Aug, Autumn = Sep–Nov) and, **per season**, report:

- the **average daily wind contribution** (% of generation) and the specific day
  closest to that seasonal average,
- the average of the **top 10%** and **bottom 10%** wind days (ranked by each
  day's wind total within that season), plus the specific day in each decile
  whose wind contribution most closely matches that decile's average.

Daily wind contribution = `100 × Σ(WIND + WIND_EMB) / Σ(GENERATION)` over the
day. The 12 representative days (3 per season) are exactly the days the LCA runs
on when `GRID_TIME_MODE = "year_average"` — each is collapsed into one
energy-weighted daily-average grid mix, so a year produces 12 results. Results
are stored in `season_wind_summary` / `summary_df` / `rep_days`.


In [ ]:
import importlib
import numpy as np
import pandas as pd
importlib.reload(H)  # pick up edits to lca_helpers.py

# --- Settings ----------------------------------------------------------------
# Defaults come from dashboard_config so this matches GRID_TIME_MODE="year_average".
ANALYSIS_YEAR = int(getattr(cfg, "GRID_YEAR", 2023))     # year to analyse
WIND_MW_COLS  = list(getattr(cfg, "GRID_WIND_COLS", ["WIND", "WIND_EMB"]))
DECILE        = float(getattr(cfg, "GRID_REP_DECILE", 0.10))

# Reuse df if it was already loaded earlier in this notebook; else load it.
try:
    df
except NameError:
    df = H.load_grid_csv()

# Shared helper does the seasonal split + representative-day selection so the
# 12 days below are identical to those used by GRID_TIME_MODE="year_average".
season_wind_summary, daily = H.seasonal_wind_summary(
    df, ANALYSIS_YEAR, wind_cols=WIND_MW_COLS, decile=DECILE
)

print(f"Seasonal wind-contribution analysis for {ANALYSIS_YEAR}")
print(f"Wind = sum of {WIND_MW_COLS};  contribution = % of GENERATION")
print("=" * 78)

for season in H.SEASON_ORDER:
    info = season_wind_summary.get(season)
    if not info:
        print(f"\n{season}: no data.")
        continue
    print(f"\n{season}  ({info['n_days']} days)")
    print("-" * 78)
    print(f"  Season average wind            : {info['season_avg_pct']:6.2f}%")
    print(f"    closest day                  : {info['rep_day'].date()}  ({info['rep_day_pct']:5.2f}%)")
    print(f"  Top 10% wind days (n={info['top10_n']:>2})        : {info['top10_avg_pct']:6.2f}%")
    print(f"    closest day                  : {info['top10_rep_day'].date()}  ({info['top10_rep_day_pct']:5.2f}%)")
    print(f"  Bottom 10% wind days (n={info['bottom10_n']:>2})     : {info['bottom10_avg_pct']:6.2f}%")
    print(f"    closest day                  : {info['bottom10_rep_day'].date()}  ({info['bottom10_rep_day_pct']:5.2f}%)")

# The 12 representative days (3 per season) that year_average mode runs the LCA on.
rep_days = H.representative_wind_days(df, ANALYSIS_YEAR, wind_cols=WIND_MW_COLS, decile=DECILE)
print(f"\nRepresentative days for year_average LCA ({len(rep_days)} days):")
for r in rep_days:
    print(f"  {r['season']:<7} {r['kind']:<8} {r['date'].date()}  (~{r['target_pct']:5.2f}% wind)")

summary_df = pd.DataFrame({k: {kk: (vv.date() if hasattr(vv, 'date') else vv)
                               for kk, vv in v.items()}
                           for k, v in season_wind_summary.items()}).T
summary_df.index.name = "season"
summary_df


Seasonal wind-contribution analysis for 2023
Wind = sum of ['WIND', 'WIND_EMB'];  contribution = % of GENERATION

Winter  (90 days)
------------------------------------------------------------------------------
  Season average wind            :  36.29%
    closest day                  : 2023-02-15  (36.23%)
  Top 10% wind days (n= 9)        :  57.66%
    closest day                  : 2023-12-23  (58.06%)
  Bottom 10% wind days (n= 9)     :  11.42%
    closest day                  : 2023-02-07  (10.79%)

Spring  (92 days)
------------------------------------------------------------------------------
  Season average wind            :  24.16%
    closest day                  : 2023-04-03  (24.09%)
  Top 10% wind days (n=10)        :  48.30%
    closest day                  : 2023-03-24  (48.66%)
  Bottom 10% wind days (n=10)     :   8.59%
    closest day                  : 2023-05-02  ( 8.32%)

Summer  (92 days)
--------------------------------------------------------------------------

,n_days,season_avg_pct,rep_day,rep_day_pct,top10_n,top10_avg_pct,top10_rep_day,top10_rep_day_pct,bottom10_n,bottom10_avg_pct,bottom10_rep_day,bottom10_rep_day_pct
season,,,,,,,,,,,,
Winter,90,36.290914,2023-02-15,36.229487,9,57.656038,2023-12-23,58.063546,9,11.422002,2023-02-07,10.793599
Spring,92,24.156521,2023-04-03,24.09494,10,48.30072,2023-03-24,48.660544,10,8.590952,2023-05-02,8.32103
Summer,92,23.814471,2023-07-06,24.060738,10,46.64809,2023-07-03,45.119423,10,7.887259,2023-08-31,8.186581
Autumn,91,29.557052,2023-10-15,29.513456,10,53.014226,2023-09-19,52.556508,10,7.554859,2023-09-09,7.954514
